# The manuscript's tables, from the public store

Every number in the model-evaluation tables of the manuscript is a metric stored on an evaluation node of the
deposited AiiDA archives. This notebook reads those nodes and rebuilds each printed cell, then derives the cost and
dataset tables from the run records in the same archives. It needs no GPU and no restricted data.

**What you need**

* The Python environment of `tools/setup_env.py` with AiiDA (`python3.10 tools/setup_env.py --prefix <env> --torch cpu
  --with-aiida`), and pandas.
* The open record, doi:10.5281/zenodo.23325086, downloaded into one folder: `lpno_provenance.aiida`,
  `lpno_provenance_rounds.aiida` and `lpno_provenance_evals.aiida`. Point `LPNO_DEPOSIT` at it. The archives are
  opened read-only, so nothing is imported and no AiiDA profile is needed.

`data/table_cells.csv` maps every printed cell to the evaluations behind it: one row per cell with the table, block, row
and column, the evaluation UUIDs, how they combine (`single`, `mean` over seeds, or `mean_sd`, the mean and the sample
standard deviation over seeds) and the value as printed. The metric conventions are in `lpno_repro.py`.

In [1]:
import csv
import json
import os
import statistics
import sys
from pathlib import Path

import pandas as pd

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
import lpno_repro as lr

DEPOSIT = Path(os.environ.get("LPNO_DEPOSIT", HERE.parent / "data_deposit" / "open_record"))   # default: where data_deposit/scripts/download.py puts it
CODE = HERE.parent
store = lr.Store(DEPOSIT)
cells = list(csv.DictReader(open(HERE / "data" / "table_cells.csv")))
print(f"{len(cells)} printed cells; archives opened: {[name for name, _ in store.profiles]}")

210 printed cells; archives opened: ['lpno_provenance.aiida', 'lpno_provenance_rounds.aiida', 'lpno_provenance_evals.aiida']


## 1. The model-evaluation tables

Tables of the main text: the architecture benchmark (`tab:arch-steady`), the weld paths (`tab:paths-accuracy`), beam
radius and preheat (`tab:preheat`) and the two alloys (`tab:twoalloy`); supplement: the training and scoring frames
(`tab:frame-difficulty`).

In [2]:
uuids = sorted({u for c in cells for u in c["evaluations"].split()})
nodes = store.fetch(uuids)
missing = [u for u in uuids if u not in nodes]
assert not missing, f"{len(missing)} evaluations are not in the archives, e.g. {missing[:3]}"
M = {u: lr.metrics(nodes[u]["attributes"]) for u in uuids}

rows = []
for c in cells:
    values = [M[u][c["column"]] for u in c["evaluations"].split()]
    got = lr.aggregate(values, c["aggregate"], lr.decimals(c["printed"]))
    rows.append({**c, "reproduced": got, "match": got == c["printed"]})
df = pd.DataFrame(rows)
print(f"{int(df.match.sum())} of {len(df)} printed cells reproduced from {len(uuids)} evaluations "
      f"({pd.Series([nodes[u]['archive'] for u in uuids]).value_counts().to_dict()})")
df.loc[~df.match, ["table", "block", "row", "column", "printed", "reproduced"]]

210 of 210 printed cells reproduced from 66 evaluations ({'lpno_provenance.aiida': 66})


,table,block,row,column,printed,reproduced


In [3]:
for table in df.table.unique():
    t = df[df.table == table]
    order = list(dict.fromkeys(t.column))
    wide = t.pivot_table(index=["block", "row"], columns="column", values="reproduced", aggfunc="first", sort=False)
    print(table)
    display(wide[order])

tab:arch-steady


column                                 T MAE      T RMSE      BB MAE  \
block                    row                                           
Quasi-steady formulation FNO     7.71 ± 0.05  29.6 ± 0.8  15.0 ± 1.1   
                         RIGNO  11.36 ± 1.69  40.0 ± 9.7  23.6 ± 2.3   
                         GAOT    9.74 ± 0.44  32.3 ± 0.9  16.8 ± 0.2   
Laboratory frame         FNO    10.01 ± 0.10  27.3 ± 1.1  49.1 ± 2.7   
                         RIGNO   6.65 ± 0.46  21.4 ± 0.5  30.9 ± 0.5   
                         GAOT   11.13 ± 0.09  41.7 ± 5.3  72.8 ± 2.6   

column                            Rel. err.  Melt-pool IoU  
block                    row                                
Quasi-steady formulation FNO    1.25 ± 0.01  0.920 ± 0.001  
                         RIGNO  1.84 ± 0.27  0.926 ± 0.004  
                         GAOT   1.58 ± 0.07  0.915 ± 0.005  
Laboratory frame         FNO    2.25 ± 0.02  0.863 ± 0.002  
                         RIGNO  1.49 ± 0.10  0.926 ± 0.001  
                         GAOT   2.50 ± 0.02  0.826 ± 0.004

tab:paths-accuracy


column                                      T MAE T RMSE BB MAE Rel. err.  \
block                                row                                    
Variable-length free-form paths      FNO    18.45  57.92  106.9      4.87   
                                     RIGNO  16.48  45.94  118.2      4.35   
                                     GAOT   14.10  40.92  100.9      3.72   
Variable speed along free-form paths FNO     7.35  29.60   95.8      2.20   
                                     RIGNO  11.44  41.89  164.5      3.42   
                                     GAOT    7.56  27.37   96.1      2.26   
Multiple beams, free-form paths      FNO    17.31  61.31   97.5      5.12   
                                     RIGNO  22.76  71.30  111.5      6.74   
                                     GAOT   17.65  59.95   94.6      5.22   

column                                     Melt-pool IoU  
block                                row                  
Variable-length free-form paths      FNO           0.723  
                                     RIGNO         0.733  
                                     GAOT          0.772  
Variable speed along free-form paths FNO           0.741  
                                     RIGNO         0.751  
                                     GAOT          0.773  
Multiple beams, free-form paths      FNO           0.471  
                                     RIGNO         0.469  
                                     GAOT          0.508

tab:preheat


column                       T MAE T RMSE BB MAE Rel. err. Melt-pool IoU
block                  row                                              
Beam-radius variation  FNO     8.5   30.9   32.6      2.05         0.923
                       RIGNO   9.4   42.4   30.9      2.26         0.932
                       GAOT    6.7   26.6   27.8      1.62         0.930
Preheat and beam shape FNO    21.0   47.9   38.2      3.18         0.823
                       RIGNO  11.4   37.0   26.9      1.72         0.867
                       GAOT   41.9   63.4   44.0      6.35         0.837

tab:twoalloy


column                                        T MAE T RMSE BB MAE Rel. err.  \
block                   row                                                   
IN718 only test set     FNO, Ti-6Al-4V only  124.38  284.1    456     30.02   
                        FNO, maximin half     13.64   41.2     84      3.31   
                        FNO, bandit half      14.00   44.2     88      3.40   
                        FNO, joint            12.79   40.8     83      3.10   
                        RIGNO, joint          13.55   40.8     78      3.29   
                        GAOT, joint           12.86   42.2     92      3.12   
Ti-6Al-4V only test set FNO, Ti-6Al-4V only   10.97   38.7     47      2.41   
                        FNO, maximin half      9.89   35.0     42      2.17   
                        FNO, bandit half      10.08   35.3     42      2.21   
                        FNO, joint            10.47   35.9     43      2.30   
                        RIGNO, joint          14.40   38.0     51      3.16   
                        GAOT, joint           11.90   39.8     55      2.62   

column                                      Melt-pool IoU  
block                   row                                
IN718 only test set     FNO, Ti-6Al-4V only         0.350  
                        FNO, maximin half           0.831  
                        FNO, bandit half            0.833  
                        FNO, joint                  0.843  
                        RIGNO, joint                0.835  
                        GAOT, joint                 0.837  
Ti-6Al-4V only test set FNO, Ti-6Al-4V only         0.903  
                        FNO, maximin half           0.903  
                        FNO, bandit half            0.906  
                        FNO, joint                  0.899  
                        RIGNO, joint                0.902  
                        GAOT, joint                 0.887

tab:frame-difficulty


column                            T MAE T RMSE BB MAE Rel. err. Melt-pool IoU
block row                                                                    
FNO   Quasi-steady, native         7.71   29.6   15.0      1.25         0.920
      Laboratory, native          10.01   27.3   49.1      2.25         0.863
      Laboratory -> quasi-steady  17.91   40.3   53.2      2.92         0.866
RIGNO Quasi-steady, native        11.36   40.0   23.6      1.84         0.926
      Laboratory, native           6.65   21.4   30.9      1.49         0.926
      Laboratory -> quasi-steady  11.50   28.6   32.7      1.88         0.935
GAOT  Quasi-steady, native         9.74   32.3   16.8      1.58         0.915
      Laboratory, native          11.13   41.7   72.8      2.50         0.826
      Laboratory -> quasi-steady  20.01   52.2   78.9      3.27         0.841

One choice behind these rows is recorded in the evaluations themselves:

* The laboratory-frame rows of `tab:arch-steady` (and the native laboratory rows of `tab:frame-difficulty`) are the
  uniform-drawing re-run models, each scored on every frame of the test split at full resolution.

In [4]:
pre = {u for c in cells if c["table"] == "tab:preheat" for u in c["evaluations"].split()}
for u in sorted(pre):
    profile = nodes[u]["attributes"]["evaluation_metadata"].get("evaluation_profile") or {}
    assert profile.get("inference_batch_size") == 1, u
print(f"tab:preheat: all {len(pre)} evaluations ran at inference batch size 1")

lab = {u for c in cells if c["table"] == "tab:arch-steady" and c["block"] == "Laboratory frame"
       and c["row"] in ("FNO", "GAOT") for u in c["evaluations"].split()}
frames = {nodes[u]["attributes"]["evaluation_metadata"]["metrics"]["T"]["tail_deficit_n_frames"] for u in lab}
print(f"laboratory FNO and GAOT: {len(lab)} evaluations, frames scored {sorted(frames)}, from "
      f"{sorted({nodes[u]['archive'] for u in lab})}")

tab:preheat: all 6 evaluations ran at inference batch size 1
laboratory FNO and GAOT: 6 evaluations, frames scored [5773.0], from ['lpno_provenance.aiida']


## 2. Training, inference and data-generation cost (`tab:cost`)

* **Training** was timed on one NVIDIA V100, one run on the GPU at a time. Each tabled seed-42 model of
  `tab:arch-steady`, quasi-steady and laboratory frame, had its own training profile re-run through `aiida-pino` for a
  warm-up and 100 timed steps, with its configuration, overrides and simulations. The laboratory-frame models train at
  the read-time stride 2,2,1 their profiles set (45 x 20 x 40 cells). The quasi-steady models read the moving-frame,
  30-frame-averaged collection with a 10-cell top crop, derived again from the same simulations by the current code:
  61 x 40 x 30 cells, one more along x than the 60 x 40 x 30 of the archived runs. The time per step is the median of
  PhysicsNeMo's `time/iteration` over the timed steps, which hold no validation or checkpoint; start-up is excluded.
  The 32,000-step figure multiplies it by 32,000, the length of every tabled training run.
* **Inference** was timed on the same GPU with `lpno/scripts/measure_inference_time.py`: a batch-one forward pass of
  each tabled laboratory-frame model on one stored frame of the test split, at full resolution.
* **Data generation** sums the solver wall time recorded on each of the 123 simulations of the straight-path set.

The measurement files in `lpno/measurements/` name, for every row, the model label and UUID it was timed from, the
steps timed and every logged value.

In [5]:
M = CODE / "lpno" / "measurements"
(train_file,) = sorted(M.glob("training_timing_*.json"))
(infer_file,) = sorted(M.glob("inference_timing_*.json"))
train, infer = json.load(open(train_file)), json.load(open(infer_file))

cost, models = {}, []
for r in train["runs"]:
    ms = r["ms_per_step_median"]
    cost.setdefault(r["architecture"], {}).update({
        f"Train {r['formulation']}, per step (ms)": round(ms),
        f"Train {r['formulation']}, 32,000 steps (h)": round(ms * 32000 / 3.6e6, 2),
    })
    models.append({"architecture": r["architecture"], "timing": f"training, {r['formulation']}",
                   "model": r["model_label"], "uuid": r["model_uuid"],
                   "steps timed": "-".join(map(str, r["steps_timed"]))})
for r in infer["results"]:
    arch = r["model_family"].upper()
    cost[arch]["Inference, per frame (ms)"] = round(r["per_frame_ms"], 1)
    models.append({"architecture": arch, "timing": "inference", "model": r["model_label"], "uuid": r["model_uuid"],
                   "steps timed": f"{r['timed_runs']} passes"})
print(f"training: {train['gpu']}, {train['exclusive']}; inference: {infer['host']['gpu']['name']}, "
      f"case {infer['test_case']['case_name']} of the test split")
display(pd.DataFrame(cost).T)
display(pd.DataFrame(models).set_index(["architecture", "timing"]).sort_index())

training: Tesla V100-PCIE-32GB, one run on the GPU at a time; inference: Tesla V100-PCIE-32GB, case kh40_000 of the test split


,"Train quasi-steady, per step (ms)","Train quasi-steady, 32,000 steps (h)","Train laboratory, per step (ms)","Train laboratory, 32,000 steps (h)","Inference, per frame (ms)"
FNO,55.0,0.49,76.0,0.67,6.8
GAOT,118.0,1.05,467.0,4.15,61.7
RIGNO,188.0,1.67,1210.0,10.76,244.6


model  \
architecture timing                                                      
FNO          inference                          uni_fno_lab_b8_32k_s42   
             training, laboratory               uni_fno_lab_b8_32k_s42   
             training, quasi-steady             fno_spkh123_steady_32k   
GAOT         inference                         uni_gaot_lab_b8_32k_s42   
             training, laboratory              uni_gaot_lab_b8_32k_s42   
             training, quasi-steady            gaot_spkh123_steady_32k   
RIGNO        inference                        uni_rigno_lab_b8_32k_s42   
             training, laboratory             uni_rigno_lab_b8_32k_s42   
             training, quasi-steady  rigno_spkh123_steady_32k_fill3520   

                                                                     uuid  \
architecture timing                                                         
FNO          inference               7d385883-011a-41a2-85a2-2aed4ba0e3ef   
             training, laboratory    7d385883-011a-41a2-85a2-2aed4ba0e3ef   
             training, quasi-steady  155020c6-d598-4641-9314-b586907cfa09   
GAOT         inference               d4bec77c-5613-42d2-b310-f5a0732665df   
             training, laboratory    d4bec77c-5613-42d2-b310-f5a0732665df   
             training, quasi-steady  3ab4f902-f816-4cfa-b490-ff4fb06fa769   
RIGNO        inference               3d91eb45-eea6-456c-873a-188cae329056   
             training, laboratory    3d91eb45-eea6-456c-873a-188cae329056   
             training, quasi-steady  21e5e326-7e5a-4e49-bd8f-44375e1f7ff3   

                                    steps timed  
architecture timing                              
FNO          inference                20 passes  
             training, laboratory        75-175  
             training, quasi-steady      25-125  
GAOT         inference                20 passes  
             training, laboratory        75-175  
             training, quasi-steady      25-125  
RIGNO        inference                20 passes  
             training, laboratory        75-175  
             training, quasi-steady      25-125

In [6]:
def generation(orm):
    coll = orm.QueryBuilder().append(orm.Node, filters={"uuid": {"like": "1d13f61f%"}}).one()[0]
    members = coll.base.attributes.get("member_simulation_uuids")
    qb = orm.QueryBuilder().append(orm.Node, filters={"uuid": {"in": members}}, project=["extras"])
    return coll.label, [e for (e,) in qb.iterall()]


label, extras = store.query("lpno_provenance.aiida", generation)
wall = sum(e["wall_seconds"] for e in extras)
cores = {e["cores"] for e in extras}
print(f"{label}: {len(extras)} simulations, {wall / 3600:.1f} h wall, "
      f"{sum(e['wall_seconds'] * e['cores'] for e in extras) / 3600:.0f} core-hours ({sorted(cores)} cores each), "
      f"{wall / len(extras) / 60:.1f} min per simulation")

ti64_straight_path_calibrated_with_keyholes: 123 simulations, 28.1 h wall, 281 core-hours ([10] cores each), 13.7 min per simulation


## 3. The datasets (`tab:datasets`)

Simulation counts and the ranges of the varied parameters, read from the case manifest each simulation node stores.
The deposit groups the simulations by family (`lpno/simulations/<family>`); `payload_manifest.csv` of the open record
gives each simulation's split, and a dataset counts the simulations its models were trained, validated and tested on
(a family can hold further simulations that no model used). The beam-radius sweep joins two families and leaves out
seven simulations of the first that nearly duplicate others (split `duplicate_excluded`). The two-alloy row counts the 272 IN718 simulations
used in training and testing: 248 acquired in the loop and the 24-case test set. Seventeen others (one test
candidate and 16 probes) were run but not used, and are not deposited. Path lengths are per beam for the
multi-beam set. For the variable-speed set the manuscript gives the range of the speed profile along each path; the
manifest's `scan_speed_m_per_s` printed here is each path's nominal speed.

In [7]:
FAMILIES = {
    "Variable paths": ["ti64_rand150_newphys"],
    "Variable speed": ["ti64_varspeed_newphys"],
    "Preheat sweep": ["ti64_preheat60"],
    "Beam-radius sweep": ["ti64_anchorbox_lhs150", "ti64_edge75_highenthalpy"],
    "Active-learning two-alloy (IN718)": ["in718_loop_", "in718_testset_twoalloy"],
    "Multi-beam": ["ti64_multibeam40_capped"],
}


def record(n):
    sm = n.base.attributes.get("simulation_metadata", {})
    return {"uuid": n.uuid, "manifest": sm.get("flow3d_case_manifest", {}), "case": sm.get("case_name")}


def family_records(orm):
    groups = {g.label.split("/", 2)[2]: g for g in orm.QueryBuilder().append(
        orm.Group, filters={"label": {"like": "lpno/simulations/%"}}).all(flat=True)}
    # the power and scan speed grid is the 123-simulation collection the models trained and were scored on
    coll = orm.QueryBuilder().append(orm.Node, filters={"uuid": {"like": "1d13f61f%"}}).one()[0]
    grid = orm.QueryBuilder().append(orm.Node, filters={"uuid": {"in": coll.base.attributes.get("member_simulation_uuids")}})
    out = {"Power scan speed grid": [record(n) for n in grid.all(flat=True)]}
    for row, fams in FAMILIES.items():
        recs = []
        for g_label, g in groups.items():
            if any(g_label == f or (f.endswith("_") and g_label.startswith(f)) for f in fams):
                recs += [record(n) for n in g.nodes]
        out[row] = recs
    return out


def ranges(recs):
    def get(d, *keys):
        for k in keys:
            d = (d or {}).get(k)
        return d

    def span(vals, scale=1.0, fmt="{:.3g}"):
        vals = [v * scale for v in vals if v is not None]
        return f"{fmt.format(min(vals))}-{fmt.format(max(vals))}" if vals else ""
    P, V, r0, T0, L = [], [], [], [], []
    for rec in recs:
        lz = rec["manifest"].get("input", {}).get("laser", {})
        beams = lz.get("beams") or [lz]
        P += [b.get("power_w") for b in beams]
        V += [b.get("scan_speed_m_per_s") for b in beams]
        r0.append(lz.get("beam_radius_m"))
        T0.append(get(rec["manifest"], "input", "simulation", "initial_temperature_k"))
        if lz.get("beams"):          # multi-beam: one path per beam
            L += [get(b, "path", "length_m") for b in lz["beams"]]
        else:
            L.append(get(rec["manifest"], "derived", "scan_path_length_m"))
    return {"P (W)": span(P), "V (m/s)": span(V), "r0 (um)": span(r0, 1e6), "T0 (K)": span(T0),
            "path length (mm)": span(L, 1e3)}


split = {r["uuid"]: r["split"] for r in csv.DictReader(open(DEPOSIT / "payload_manifest.csv"))}
records = store.query("lpno_provenance.aiida", family_records)
PRINTED = {   # the manuscript's table, for comparison
    "Power scan speed grid": "123; P 40-215 W, V 0.10-0.96 m/s",
    "Variable paths": "150; length 0.11-2.97 mm, P 68-245 W, V 0.21-0.99 m/s",
    "Variable speed": "140; length 0.20-1.63 mm, speed profile 0-2.0 m/s, P 53-252 W",
    "Preheat sweep": "107; T0 296-761 K, P 55-284 W, V 0.20-1.19 m/s, r0 35-80 um",
    "Beam-radius sweep": "218; P 90-235 W, V 0.12-1.25 m/s, r0 42.5-65 um",
    "Active-learning two-alloy (IN718)": "272; P 90-350 W, V 0.15-1.25 m/s, r0 22-65 um",
    "Multi-beam": "79; P 27-155 W per beam, V 0.22-0.98 m/s",
}
table = {}
for row in ["Power scan speed grid", *FAMILIES]:     # the manuscript's order
    kept = [r for r in records[row] if split.get(r["uuid"]) in ("train", "val", "test")]
    table[row] = {"simulations": len(kept), **ranges(kept), "printed": PRINTED[row]}
display(pd.DataFrame(table).T)

,simulations,P (W),V (m/s),r0 (um),T0 (K),path length (mm),printed
Power scan speed grid,123,40-215,0.101-0.96,50-50,273-273,0.597-0.597,"123; P 40-215 W, V 0.10-0.96 m/s"
Variable paths,150,67.8-245,0.205-0.994,50-50,293-293,0.107-2.97,"150; length 0.11-2.97 mm, P 68-245 W, V 0.21-0..."
Variable speed,140,53.3-252,0.15-1.25,50-50,293-293,0.204-1.63,"140; length 0.20-1.63 mm, speed profile 0-2.0 ..."
Preheat sweep,107,55.5-284,0.2-1.19,35.5-79.7,296-761,0.597-0.597,"107; T0 296-761 K, P 55-284 W, V 0.20-1.19 m/s..."
Beam-radius sweep,218,90-235,0.12-1.25,42.5-65,273-273,0.299-0.597,"218; P 90-235 W, V 0.12-1.25 m/s, r0 42.5-65 um"
Active-learning two-alloy (IN718),272,90-350,0.15-1.25,22-65,273-273,0.597-0.597,"272; P 90-350 W, V 0.15-1.25 m/s, r0 22-65 um"
Multi-beam,79,26.7-155,0.216-0.982,47.5-47.5,293-293,0.242-2.89,"79; P 27-155 W per beam, V 0.22-0.98 m/s"


## 4. Tables that rest on the restricted record or on published values

* `tab:closing-conditions` (supplement): measured depths from the published experiments (`experimental_data/`), the
  FLOW-3D and surrogate depths read from the deposited simulations and the joint RIGNO's frame dumps, both in the
  restricted record. The depths are committed in `figures/data/{ti64,in718}_joint_rigno_alloyflag_depths.json`
  and printed here from those files.
* `tab:fluctuation` (supplement): statistics of the ground-truth temperature over time, computed from the restricted
  payloads with `lpno/scripts/fluctuation_floor.py`.
* `tab:benchmark` (supplement): the pre-validation benchmark. Its models were trained on the earlier solver settings
  and are not in the deposit; the LP-FNO row is the published cross-validation mean of the predecessor study.

In [8]:
FIG = CODE / "reproducibility" / "figures" / "data"
ti = json.load(open(FIG / "ti64_joint_rigno_alloyflag_depths.json"))
inc = json.load(open(FIG / "in718_joint_rigno_alloyflag_depths.json"))
rows = [{"alloy": "Ti-6Al-4V", "case": r["case"], "P (W)": r["P_w"], "V (m/s)": r["V_mps"], "r0 (um)": r["r0_um"],
         "kind": r["kind"], "measured (um)": r["meas_d"], "FLOW-3D (um)": r["sim_d"], "RIGNO (um)": r["model_d"]}
        for r in ti["rows"] if r["case"] != "abx_010_galedge100"]
rows += [{"alloy": "IN718", "case": r["case"], "P (W)": r["P_w"], "V (m/s)": r["V_mps"], "r0 (um)": r["spot_um"] / 2,
          "kind": "fusion", "measured (um)": r["meas_depth_um"], "FLOW-3D (um)": r["flow3d_depth_um"],
          "RIGNO (um)": r["surrogate_depth_um"]} for r in inc["rows"]]
print(f"model {ti['model_label']}; depths from its frame dumps (melt-pool depth: liquid fraction > 0.5; cavity depth: "
      f"the gas-metal interface), measured depths from the published experiments")
display(pd.DataFrame(rows))

model rigno_joint_r05_alloyflag_s42; depths from its frame dumps (melt-pool depth: liquid fraction > 0.5; cavity depth: the gas-metal interface), measured depths from the published experiments


,alloy,case,P (W),V (m/s),r0 (um),kind,measured (um),FLOW-3D (um),RIGNO (um)
0,Ti-6Al-4V,abx_000_gal225,225.000,0.40,60.0,fusion,242.6000,230.000,220.0
1,Ti-6Al-4V,abx_001_gan206,206.395,0.40,47.5,depression,226.7660,235.895,230.0
2,Ti-6Al-4V,abx_004_vsw209_11,209.000,1.10,47.5,depression,79.4804,65.294,60.0
3,Ti-6Al-4V,abx_005_lad_r60_v09,209.000,0.90,60.0,fusion,52.9000,70.000,70.0
4,Ti-6Al-4V,abx_007_vsw209_09,209.000,0.90,47.5,depression,99.4804,87.143,80.0
5,Ti-6Al-4V,abx_008_vsw156_07,156.000,0.70,47.5,depression,61.6388,62.593,60.0
6,Ti-6Al-4V,abx_009_vsw156_06,156.000,0.60,47.5,depression,76.6388,80.318,80.0
7,Ti-6Al-4V,abx_011_vsw208_07,208.000,0.70,47.5,depression,116.7700,120.741,110.0
8,Ti-6Al-4V,abx_012_vsw208_06,208.000,0.60,47.5,depression,119.2700,142.064,140.0
9,Ti-6Al-4V,abx_013_gal150,150.000,0.40,60.0,fusion,141.8200,100.000,100.0
